# Demo tích hợp Chương 3 — Frequent Itemset Mining (negFIN)

Notebook tổng hợp hai ứng dụng: **Market Basket Analysis** và **Bag of Visual Words**, dùng trực tiếp `src/algorithm/negfin_baseline.jl` của nhóm. Chạy lần lượt bằng **Runtime → Run all** sau khi đã push code lên GitHub. Kết quả được ghi dưới `results/chapter3/`.


In [ ]:
# Tự dùng code trong repository hoặc clone repo chính thức (sau khi đã push Chương 3).
from pathlib import Path
import os, subprocess, sys, shutil, platform

REPO_URL = "https://github.com/Thanhtung46/Frequent_Itemset_Mining.git"
PROJECT_DIR = None
for directory in [Path.cwd(), Path.cwd() / 'Frequent_Itemset_Mining', Path('/content/Frequent_Itemset_Mining')]:
    if (directory / 'src' / 'algorithm' / 'negfin_baseline.jl').exists() and (directory / 'applications').is_dir():
        PROJECT_DIR = directory.resolve()
        break
if PROJECT_DIR is None:
    if not REPO_URL.startswith('https://github.com/'):
        raise ValueError('Đặt REPO_URL tới repo GitHub public của nhóm.')
    PROJECT_DIR = Path('/content/Frequent_Itemset_Mining')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
if not (PROJECT_DIR / 'applications' / 'common.py').exists():
    raise RuntimeError('Repo GitHub chưa được push phần Chương 3. Hãy commit/push mã nguồn mới.')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Project directory:', PROJECT_DIR)


In [ ]:
# Cài gói Python hỗ trợ đọc dữ liệu/biểu đồ/ảnh; KHÔNG cài gói khai thác FIM.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-ch3.txt'], check=True)

# Cài Julia nếu đang dùng Colab Linux. Máy Windows/Mac: cài Julia >= 1.9 trước.
if shutil.which(os.environ.get('JULIA_BIN', 'julia')) is None:
    if sys.platform == 'linux' and platform.machine() in ('x86_64', 'AMD64'):
        julia_home = Path('/content/julia-1.11.4') if Path('/content').is_dir() else Path.home() / '.cache' / 'negfin-julia-1.11.4'
        julia_exe = julia_home / 'bin' / 'julia'
        if not julia_exe.exists():
            command = ('curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/'
                       'julia-1.11.4-linux-x86_64.tar.gz -o /tmp/negfin-julia.tar.gz'
                       ' && mkdir -p ' + str(julia_home)
                       + ' && tar -xzf /tmp/negfin-julia.tar.gz -C ' + str(julia_home)
                       + ' --strip-components=1')
            subprocess.run(['bash', '-lc', command], check=True)
        os.environ['JULIA_BIN'] = str(julia_exe)
    else:
        raise RuntimeError('Cần cài Julia >= 1.9 và đưa julia vào PATH hoặc đặt biến JULIA_BIN.')
subprocess.run([os.environ.get('JULIA_BIN', 'julia'), '--version'], check=True)
print('Dùng negFIN của nhóm:', PROJECT_DIR / 'src' / 'algorithm' / 'negfin_baseline.jl')

## 1. Kiểm thử chuẩn bị dữ liệu Python và thuật toán Julia

Kiểm thử Julia có thể thất bại nếu cài đặt negFIN của nhóm còn lỗi; không thay thuật toán bằng thư viện FIM khác.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_ch3.py'], check=True)
subprocess.run([os.environ.get('JULIA_BIN', 'julia'), '--project=.', '--startup-file=no', 'tests/test_ch3_julia.jl'], check=True)


## 2. Ứng dụng phân tích giỏ hàng

Luật kết hợp được xếp theo lift; cần đọc đồng thời support và confidence.

In [ ]:
from applications.run_market import execute as run_market
from IPython.display import display, Image
market = run_market(minsup=0.01, minconf=0.15)
display(market['rules'].head(10))
display(Image(filename=str(market['work_dir'] / 'top_products.png')))
if (market['work_dir'] / 'top10_lift.png').exists():
    display(Image(filename=str(market['work_dir'] / 'top10_lift.png')))


## 3. Ứng dụng khai thác mẫu trực quan (Bag of Visual Words)

Chia train/test, dùng dense SIFT và K-means tạo giao dịch; tìm frequent itemsets từ negFIN Julia; so sánh độ chính xác của Logistic Regression.

In [ ]:
from applications.run_visual import execute as run_visual
visual = run_visual(minsup=0.08, n_words=24)
display(visual['enrichment'].head(10))
display(visual['evaluation'])
display(Image(filename=str(visual['work_dir'] / 'sample_digits.png')))
if (visual['work_dir'] / 'enriched_patterns.png').exists():
    display(Image(filename=str(visual['work_dir'] / 'enriched_patterns.png')))


## 4. Viết báo cáo

Dẫn nguồn dữ liệu, thuật toán negFIN và cách chọn ngưỡng; lấy bảng/ảnh *thực sự sinh ra sau khi chạy* trong `results/chapter3/`. Không tự điền số liệu khi chưa kiểm thử Julia. Việc lớp ảnh ít giúp demo dễ tái lập, nhưng phải mô tả rõ giới hạn so với Caltech-101/CIFAR.
